# Le boosting pas à pas : prédire le départ de clients

Un opérateur télécom a perdu, le mois dernier, plus d'un quart de ses clients. Il voudrait repérer à l'avance ceux qui risquent de partir (le *churn*) pour leur proposer une offre de fidélisation.

Au programme :

1. Voir le boosting corriger ses erreurs, arbre après arbre, sur une seule variable
2. Comparer un arbre de décision, une forêt aléatoire et du boosting (CatBoost) sur toutes les variables
3. Comprendre les prédictions du modèle, et jouer avec

Pas besoin de GPU : tout le notebook tourne en moins d'une minute.

Données : [Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) (IBM, licence Apache 2.0), 7 043 clients fictifs.

## Installation et imports

In [ ]:
!pip install --quiet catboost

import time

import ipywidgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from matplotlib.ticker import PercentFormatter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

## Les données

In [ ]:
customers = pd.read_csv("https://raw.githubusercontent.com/shuuchuu/datasets/main/telco-churn/telco-churn.csv")

# Noms de colonnes et valeurs en français
columns = {
  "gender": "Genre",
  "SeniorCitizen": "Senior",
  "Partner": "En couple",
  "Dependents": "Personnes à charge",
  "tenure": "Ancienneté (mois)",
  "PhoneService": "Téléphone",
  "MultipleLines": "Plusieurs lignes",
  "InternetService": "Internet",
  "OnlineSecurity": "Sécurité en ligne",
  "OnlineBackup": "Sauvegarde en ligne",
  "DeviceProtection": "Protection des appareils",
  "TechSupport": "Support technique",
  "StreamingTV": "TV en streaming",
  "StreamingMovies": "Films en streaming",
  "Contract": "Contrat",
  "PaperlessBilling": "Facture dématérialisée",
  "PaymentMethod": "Moyen de paiement",
  "MonthlyCharges": "Facture mensuelle ($)",
  "TotalCharges": "Total facturé ($)",
  "Churn": "Parti",
}
values = {
  "Yes": "Oui",
  "No": "Non",
  "Male": "Homme",
  "Female": "Femme",
  "No phone service": "Pas de téléphone",
  "No internet service": "Pas d'internet",
  "Fiber optic": "Fibre",
  "DSL": "ADSL",
  "Month-to-month": "Sans engagement",
  "One year": "1 an",
  "Two year": "2 ans",
  "Electronic check": "Chèque électronique",
  "Mailed check": "Chèque postal",
  "Bank transfer (automatic)": "Virement automatique",
  "Credit card (automatic)": "Carte bancaire automatique",
}

customers = customers.drop(columns="customerID").rename(columns=columns).replace(values)
customers["Senior"] = customers["Senior"].map({0: "Non", 1: "Oui"})
# Les nouveaux clients n'ont encore rien payé : leur total est vide
customers["Total facturé ($)"] = pd.to_numeric(customers["Total facturé ($)"], errors="coerce").fillna(0)
customers["Parti"] = (customers["Parti"] == "Oui").astype(int)

print(f"{len(customers)} clients, dont {customers['Parti'].mean():.1%} sont partis")
customers.head()

Un premier regard : le taux de départ selon quelques variables.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5), sharex=True)
for ax, column in zip(axes, ["Contrat", "Internet", "Moyen de paiement"]):
  rates = customers.groupby(column)["Parti"].mean().sort_values()
  ax.barh(rates.index, rates.to_numpy(), color="tab:red")
  ax.set_title(column)
  ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
axes[1].set_xlabel("Taux de départ")
fig.tight_layout()

On met des clients de côté pour évaluer honnêtement les modèles :

- **entraînement** (60 %) : les modèles apprennent sur ces clients ;
- **validation** (20 %) : on choisit les réglages (par exemple le nombre d'arbres) sur ces clients ;
- **test** (20 %) : on mesure le résultat final sur ces clients, jamais vus jusque-là.

In [ ]:
train, rest = train_test_split(customers, test_size=0.4, stratify=customers["Parti"], random_state=0)
validation, test = train_test_split(rest, test_size=0.5, stratify=rest["Parti"], random_state=0)
print(f"Entraînement : {len(train)}, validation : {len(validation)}, test : {len(test)}")

## 1. Le boosting pas à pas

Pour voir ce que fait le boosting, on ne garde d'abord qu'une variable : la facture mensuelle. Le modèle doit prédire la probabilité qu'un client parte à partir de sa seule facture.

On écrit le boosting nous-mêmes, en quelques lignes :

1. on part d'une prédiction naïve : le taux de départ moyen, pour tout le monde ;
2. un petit arbre (2 niveaux de questions) apprend les **erreurs** de cette prédiction ;
3. on ajoute une fraction de sa correction à la prédiction (le « pas » de la descente de gradient) ;
4. un nouvel arbre apprend les erreurs restantes, et ainsi de suite.

In [ ]:
BILL = "Facture mensuelle ($)"
STEP = 0.3  # chaque arbre ne corrige qu'une partie de l'erreur
N_TREES = 200

x, y = train[[BILL]].to_numpy(), train["Parti"].to_numpy()
x_val, y_val = validation[[BILL]].to_numpy(), validation["Parti"].to_numpy()

trees = []
predictions = [np.full(len(y), y.mean())]  # prédiction naïve : le taux moyen
predictions_val = [np.full(len(y_val), y.mean())]
for _ in range(N_TREES):
  errors = y - predictions[-1]
  tree = DecisionTreeRegressor(max_depth=2).fit(x, errors)  # l'arbre apprend les erreurs
  trees.append(tree)
  predictions.append(predictions[-1] + STEP * tree.predict(x))
  predictions_val.append(predictions_val[-1] + STEP * tree.predict(x_val))

# Erreur moyenne (au carré) après 0, 1, 2… arbres
train_error = [np.mean((y - p) ** 2) for p in predictions]
val_error = [np.mean((y_val - p) ** 2) for p in predictions_val]

Chaque point est le taux de départ observé dans une tranche de facture de 5 $. À gauche, le modèle après quelques arbres ; à droite, ses erreurs dans chaque tranche, et l'arbre suivant qui apprend à les corriger.

In [ ]:
grid = np.linspace(x.min(), x.max(), 500).reshape(-1, 1)
curves = np.cumsum([np.full(len(grid), y.mean())] + [STEP * t.predict(grid) for t in trees], axis=0)

brackets = pd.cut(train[BILL], bins=np.arange(15, 125, 5))
centers = np.array([b.mid for b in brackets.cat.categories])
in_bracket = brackets.cat.codes.to_numpy()
observed_rate = np.array([y[in_bracket == i].mean() if (in_bracket == i).any() else np.nan for i in range(len(centers))])


def plot_data(ax):
  ax.scatter(centers, observed_rate, color="tab:red", s=15, label="Taux de départ observé")
  ax.set_xlabel(BILL)
  ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))


def show_step(n):
  fig, (left, right) = plt.subplots(1, 2, figsize=(14, 3.5))
  plot_data(left)
  left.plot(grid, curves[n], color="tab:blue", label=f"Modèle ({n} arbre{'s' if n > 1 else ''})")
  left.set_title(f"Modèle après {n} arbre{'s' if n > 1 else ''}")
  left.set_ylim(0, 0.6)
  left.legend(loc="upper left")
  bracket_errors = [(y - predictions[n])[in_bracket == i].mean() for i in range(len(centers))]
  right.bar(centers, bracket_errors, width=4, color="tab:gray", label="Erreur moyenne de la tranche")
  right.plot(grid, trees[n].predict(grid), color="tab:green", label=f"Arbre n° {n + 1} : correction apprise")
  right.axhline(0, color="black", linewidth=0.5)
  right.set_title("Erreurs restantes, et l'arbre suivant qui les apprend")
  right.set_xlabel(BILL)
  right.set_ylim(-0.3, 0.3)
  right.legend(loc="upper left")
  plt.show()


for n in range(3):
  show_step(n)

Et avec plus d'arbres ? Faites glisser le curseur : la courbe colle de mieux en mieux aux données d'entraînement… jusqu'à en apprendre le bruit. L'erreur sur les données de validation, elle, remonte : c'est le **sur-apprentissage**. Le bon nombre d'arbres est celui qui minimise l'erreur de validation.

In [ ]:
best = int(np.argmin(val_error))


def explore(n_trees):
  fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4))
  plot_data(left)
  left.plot(grid, curves[n_trees], color="tab:blue", label="Modèle")
  left.set_title(f"Modèle après {n_trees} arbres")
  left.set_ylim(0, 0.6)
  left.legend(loc="upper left")
  right.plot(train_error, label="Entraînement")
  right.plot(val_error, label="Validation")
  right.axvline(n_trees, color="black", linestyle="--")
  right.scatter([best], [val_error[best]], color="tab:orange", zorder=3, label=f"Meilleur : {best} arbres")
  right.set_xlabel("Nombre d'arbres")
  right.set_ylabel("Erreur moyenne")
  right.legend()
  plt.show()


ipywidgets.interact(
  explore, n_trees=ipywidgets.IntSlider(value=1, min=0, max=N_TREES, description="Arbres", continuous_update=False)
);

## 2. Arbre, forêt ou boosting ?

On utilise maintenant toutes les variables, et on compare les trois familles de modèles vues en cours sur les mêmes clients.

Les arbres de scikit-learn ne comprennent que des nombres : on transforme chaque variable catégorielle (« Contrat », « Internet »…) en colonnes 0/1. CatBoost, lui, prend ces variables telles quelles.

In [ ]:
features = [c for c in customers.columns if c != "Parti"]
categorical = customers[features].select_dtypes(exclude="number").columns.tolist()
as_numbers = pd.get_dummies(customers[features], columns=categorical, dtype=float)

y_train, y_val, y_test = train["Parti"], validation["Parti"], test["Parti"]
models, durations, probas = {}, {}, {}

start = time.perf_counter()
models["Arbre de décision"] = DecisionTreeClassifier(max_depth=5, random_state=0).fit(
  as_numbers.loc[train.index], y_train
)
durations["Arbre de décision"] = time.perf_counter() - start

start = time.perf_counter()
models["Forêt aléatoire"] = RandomForestClassifier(
  n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=0
).fit(as_numbers.loc[train.index], y_train)
durations["Forêt aléatoire"] = time.perf_counter() - start

for name in models:
  probas[name] = models[name].predict_proba(as_numbers.loc[test.index])[:, 1]

CatBoost enchaîne jusqu'à 1 000 arbres, en surveillant l'erreur sur les données de validation. Comme dans la partie 1, elle finit par remonter : CatBoost garde le modèle au meilleur nombre d'arbres.

In [ ]:
start = time.perf_counter()
catboost = CatBoostClassifier(iterations=1000, learning_rate=0.05, random_seed=0, verbose=False)
catboost.fit(
  train[features], y_train, cat_features=categorical, eval_set=(validation[features], y_val), use_best_model=True
)
durations["CatBoost"] = time.perf_counter() - start
models["CatBoost"] = catboost
probas["CatBoost"] = catboost.predict_proba(test[features])[:, 1]

history = catboost.get_evals_result()
plt.figure(figsize=(8, 3.5))
plt.plot(history["learn"]["Logloss"], label="Entraînement")
plt.plot(history["validation"]["Logloss"], label="Validation")
plt.axvline(catboost.get_best_iteration(), color="tab:orange", linestyle="--", label="Meilleur nombre d'arbres")
plt.xlabel("Nombre d'arbres")
plt.ylabel("Erreur")
plt.legend()
plt.show()

### Qui a le mieux classé les clients de test ?

On mesure l'**AUC** : prenez au hasard un client parti et un client resté ; l'AUC est la probabilité que le modèle juge le client parti plus à risque. 50 % correspond au hasard, 100 % à un modèle parfait.

In [ ]:
pd.DataFrame(
  {
    "AUC (test)": {name: f"{roc_auc_score(y_test, p):.1%}" for name, p in probas.items()},
    "Temps d'apprentissage": {name: f"{d:.1f} s" for name, d in durations.items()},
  }
)

Les écarts sont faibles ici : avec 7 000 clients et une vingtaine de variables, le problème est simple, et même un arbre seul s'en sort bien. Sur des données plus riches (plus de clients, de variables, d'interactions entre elles), l'écart en faveur du boosting est souvent plus net.

### Et pour le métier ?

L'équipe fidélisation ne peut pas appeler tout le monde. Si elle appelle les clients dans l'ordre du risque prédit, quelle part des futurs départs touche-t-elle ?

In [ ]:
CALLED = 0.2

fig, ax = plt.subplots(figsize=(7, 5))
for name, p in probas.items():
  churned = y_test.to_numpy()[np.argsort(-p)]
  share_called = np.arange(1, len(churned) + 1) / len(churned)
  share_reached = np.cumsum(churned) / churned.sum()
  ax.plot(share_called, share_reached, label=name)
  reached = share_reached[int(CALLED * len(churned)) - 1]
  print(f"{name} : en appelant {CALLED:.0%} des clients, on touche {reached:.0%} des départs")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Au hasard")
ax.axvline(CALLED, color="black", linewidth=0.5)
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel("Part des clients appelés, des plus à risque aux moins à risque")
ax.set_ylabel("Part des départs touchés")
ax.legend()
plt.show()

## 3. Comprendre le modèle, et jouer avec

Quelles variables le modèle utilise-t-il le plus ?

In [ ]:
importances = pd.Series(catboost.get_feature_importance(), index=features).sort_values()
importances.plot.barh(figsize=(7, 5), color="tab:blue", title="Importance des variables (CatBoost)")
plt.show()

Pourquoi ce client-ci est-il jugé à risque ? Pour chaque client, CatBoost sait décomposer sa prédiction en contributions de chaque variable (des [valeurs de Shapley](https://fr.wikipedia.org/wiki/Valeur_de_Shapley)) : en rouge, ce qui pousse vers le départ ; en vert, ce qui retient le client.

In [ ]:
# Un client jugé à risque, sans être le cas le plus extrême
customer = test.index[np.argsort(probas["CatBoost"])[int(0.9 * len(test))]]

contributions = catboost.get_feature_importance(
  Pool(test.loc[[customer], features], cat_features=categorical), type="ShapValues"
)[0, :-1]
contributions = pd.Series(contributions, index=[f"{f} = {test.loc[customer, f]}" for f in features])
top = contributions[contributions.abs().sort_values().index[-8:]]

ax = top.plot.barh(figsize=(8, 4), color=["tab:red" if c > 0 else "tab:green" for c in top])
ax.set_xticks([])
plt.title(f"Probabilité de départ prédite : {catboost.predict_proba(test.loc[[customer], features])[0, 1]:.0%}")
plt.xlabel("← retient le client            pousse au départ →")
plt.show()

Et si on changeait sa situation ? Modifiez son contrat, ses services ou sa facture, et regardez sa probabilité de départ évoluer.

In [ ]:
def simulate(contract, internet, tech_support, payment, tenure, bill):
  changed = test.loc[[customer], features].copy()
  changed["Contrat"] = contract
  changed["Internet"] = internet
  changed["Support technique"] = tech_support if internet != "Non" else "Pas d'internet"
  changed["Moyen de paiement"] = payment
  changed["Ancienneté (mois)"] = tenure
  changed["Facture mensuelle ($)"] = bill
  changed["Total facturé ($)"] = tenure * bill
  proba = catboost.predict_proba(changed)[0, 1]
  fig, ax = plt.subplots(figsize=(8, 0.8))
  ax.barh([0], [proba], color="tab:red" if proba > 0.5 else "tab:green")
  ax.set_xlim(0, 1)
  ax.set_yticks([])
  ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
  ax.axvline(initial, color="black", linestyle="--")
  ax.set_title(f"Probabilité de départ : {proba:.0%} (au départ : {initial:.0%})")
  plt.show()


current = test.loc[customer]
initial = probas["CatBoost"][test.index.get_loc(customer)]
ipywidgets.interact(
  simulate,
  contract=ipywidgets.Dropdown(options=["Sans engagement", "1 an", "2 ans"], value=current["Contrat"], description="Contrat"),
  internet=ipywidgets.Dropdown(options=["Fibre", "ADSL", "Non"], value=current["Internet"], description="Internet"),
  tech_support=ipywidgets.Dropdown(
    options=["Oui", "Non"], value=current["Support technique"] if current["Internet"] != "Non" else "Non", description="Support"
  ),
  payment=ipywidgets.Dropdown(
    options=sorted(customers["Moyen de paiement"].unique()), value=current["Moyen de paiement"], description="Paiement"
  ),
  tenure=ipywidgets.IntSlider(value=current["Ancienneté (mois)"], min=0, max=72, description="Ancienneté", continuous_update=False),
  bill=ipywidgets.FloatSlider(
    value=current["Facture mensuelle ($)"], min=18, max=120, step=1, description="Facture ($)", continuous_update=False
  ),
);

## À retenir

- Le boosting enchaîne de petits arbres, chacun apprenant à corriger les erreurs des précédents.
- Trop d'arbres mène au sur-apprentissage : le nombre d'arbres se choisit sur des données de validation, jamais sur les données de test.
- Sur des données tabulaires (fichiers clients, transactions…), le boosting est très souvent le meilleur choix ; CatBoost gère directement les variables catégorielles.
- Un score technique (l'AUC) se traduit en décision métier : combien de départs on touche en appelant combien de clients.
- Attention : le modèle trouve des corrélations, pas des causes. Faire signer un contrat de 2 ans à un client ne le rendra pas forcément fidèle. Et faut-il laisser l'âge (« Senior ») ou le genre influencer qui reçoit une offre ?